[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/22_conv2d.ipynb)

# 🟠 Medium: 2D Convolution

Implement **2D convolution** from scratch.

### Signature
```python
def my_conv2d(x, weight, bias=None, stride=1, padding=0):
    # x: (B, C_in, H, W), weight: (C_out, C_in, kH, kW)
    # Returns: (B, C_out, H_out, W_out)
```

### Rules
- Do NOT use `F.conv2d` or `nn.Conv2d`
- Support `stride` and `padding` parameters
- `F.pad` for zero-padding is allowed

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.7 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn.functional as F

In [20]:
def my_conv2d(x, weight, bias=None, stride=1, padding=0):
    """
    x: (B, Cin, H, W)
    weight: (Cout, Cin, kH, kW)
    bia: (Cout,)
    return (B, Cout, newH, newW)


    newH = (H - kH + 2P) / S + 1
    newW = (W - kW + 2P) / S + 1
    """
    B, Cin, H, W = x.shape

    Cout, _, kH, kW = weight.shape

    if padding > 0:
      x = F.pad(x, (padding, padding, padding, padding), 'constant', 0)

    newH = ((H - kH + 2*padding) // stride) + 1
    newW = ((W - kW + 2*padding) // stride) + 1



    #we want for each pixel (i, j)
    #patch[b, c_out, iS + u, jS + v] = sum_over_c_in-u-v(x * weight) and we want this for every combination of b and c_out (so broadcasting)
    #with u and v being the single weights of the kernel (up to kH and kW)


    out = torch.empty((B, Cout, newH, newW))

    for i in range(newH):
      for j in range(newW):
        h_start = i*stride
        w_start = j*stride

        patch = x[:, :, h_start:h_start + kH, w_start:w_start + kW]

        #         (B, 1, Cin, H, W)    (1, Cout, kH, kW)
        values = (patch.unsqueeze(1) * weight.unsqueeze(0)).sum(dim=(2, 3, 4)) #when multiplying we weill broadcast to (B, Cout, Cin, H, W) and then we need to sum over last three dims to obtain (B, Cout)
        out[:, :, i, j] = values #one value per (b, c_out) done for every i (newH) and j (newW) --> (B, Cout, newH, newW)

    if bias is not None: #bias: (Cout,) --> one per kernel
      out += bias.view(1, -1, 1, 1) #(1, Cout, 1, 1) --> broadccast over all samples, all pixels, one per kernel

    return out


In [21]:
# 🧪 Debug
x = torch.randn(1, 3, 8, 8)
w = torch.randn(16, 3, 3, 3)
print('Output:', my_conv2d(x, w).shape)
print('Match:', torch.allclose(my_conv2d(x, w), F.conv2d(x, w), atol=1e-4))

Output: torch.Size([1, 16, 6, 6])
Match: True


In [22]:
# ✅ SUBMIT
from torch_judge import check
check('conv2d')


🧪 Testing: 2D Convolution (Medium)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (2.2ms)
  ✅ [2/5] Matches F.conv2d (173.1ms)
  ✅ [3/5] With padding (13.0ms)
  ✅ [4/5] With stride (2.9ms)
  ✅ [5/5] Gradient flow (28.1ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (219.3ms total)
  Progress saved. Run status() to see your dashboard.

